To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
import os, re
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # reduces OOM from allocator fragmentation
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install snac torchcodec "datasets>=3.4.1,<4.0.0"

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

Thank you to [Etherl](https://huggingface.co/Etherll) for creating this notebook!

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "k2-fsa/OmniVoice"

# Load directly via Hugging Face with custom code execution allowed
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True  # Allows loading the custom 'omnivoice' architecture
)

ValueError: The checkpoint you are trying to load has model type `omnivoice` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.8.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a name="Data"></a>
### Data Prep  

We will use the `Etherll/kaira`, which is designed for training TTS models. Ensure that your dataset follows the required format: **text, audio** for single-speaker models or **source, text, audio** for multi-speaker models. You can modify this section to accommodate your own dataset, but maintaining the correct structure is essential for optimal training.

In [ ]:
# Dataset pipeline starts in the Thai loader cell below.
# (The old Etherll/kaira stub was removed so it cannot overwrite the mixed set.)

### Thai data with real speaker labels: `Chalermdej/yodas2_sidon_th_tts`

Zero-shot voice cloning is taught by showing the model *pairs* of clips from the same
speaker: one completed utterance is prepended as a reference turn, and the model learns
to speak the target turn in that same voice. That only works if the dataset has genuine
speaker labels **and** several clips per speaker.

`CMKL/Porjai-Thai-voice-dataset-central` cannot do this — it is crowdsourced with one
contributor per sentence and has no speaker id at all, so every row would be its own
speaker and no pair could ever be formed.

This dataset instead gives us:

| | |
|---|---|
| rows / hours | 139,088 clips, 156 h (train) |
| speakers | 4,199 real `speaker_id` values (~33 clips each) |
| sample rate | 24 kHz — already exactly what SNAC wants, no resampling |
| text | normalized Thai, verified by several ASR models |
| quality | `grade_avg` of `S+`/`S`/`A+` plus DNSMOS scores to filter on |

One caveat drives the loading code below: the rows are **shuffled across all 54 parquet
shards**, so a slice like `train[:5000]` would land ~5,000 different speakers with one
clip each and we would be back to the Porjai problem. Instead we download a handful of
whole shards and then keep only the speakers that ended up with two or more clips.

In [ ]:
#@title Load whole parquet shards (NOT a row slice - see the note above)
from datasets import load_dataset, Audio

# Measured from the dataset's metadata.parquet (each shard is ~2.6k rows / ~0.5 GB):
#   4 shards  -> ~10k rows, 2.7k speakers, 3.8 clips each, 91% of rows pairable
#   8 shards  -> ~20k rows, 3.3k speakers, 6.1 clips each, 96% of rows pairable
#  12 shards  -> ~30k rows, 3.6k speakers, 8.4 clips each, 98% of rows pairable
N_SHARDS = 1
TOTAL_SHARDS = 54

MIN_CLIPS_PER_SPK = 2    # a speaker is useless for pairing without at least a ref and a target
MAX_CLIPS_PER_SPK = 12   # cap so a few talkative speakers don't dominate the mix
MIN_DUR, MAX_DUR = 1.5, 12.0
KEEP_GRADES = {"S+", "S"} # A+ is the lowest of the three tiers

shards = [f"data/train-{i:05d}-of-{TOTAL_SHARDS:05d}.parquet" for i in range(N_SHARDS)]
raw = load_dataset(
   "Chalermdej/yodas2_sidon_th_tts",
    data_files={"train": shards},
    split="train",
    verification_mode="no_checks",
)
raw = raw.cast_column("audio", Audio(sampling_rate = 24000))
print(raw)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00054.parquet:   0%|          | 0.00/487M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/139088 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'utt_id', 'text', 'speaker_id', 'grade_avg', 'duration', 'dnsmos_overall', 'dnsmos_signal', 'dnsmos_background', 'original_ref', 'whisper_large', 'whisper_typhoon', 'typhoon_audio'],
    num_rows: 2576
})


In [ ]:
#@title Filter on quality/duration and keep speakers that have enough clips to pair
import collections, random
random.seed(3407)

# Read the metadata columns directly instead of using .filter(), which would decode
# every audio file just to look at a duration.
durations = raw["duration"]
grades    = raw["grade_avg"]
speakers  = raw["speaker_id"]

by_speaker = collections.defaultdict(list)
for i, (dur, grade, spk) in enumerate(zip(durations, grades, speakers)):
    if MIN_DUR <= dur <= MAX_DUR and grade in KEEP_GRADES:
        by_speaker[spk].append(i)

selected = []
for spk, idxs in by_speaker.items():
    if len(idxs) < MIN_CLIPS_PER_SPK:
        continue
    random.shuffle(idxs)
    selected.extend(idxs[:MAX_CLIPS_PER_SPK])
random.shuffle(selected)

dataset = (
    raw.select(selected)
       .select_columns(["audio", "text", "speaker_id"])
       .rename_column("speaker_id", "source")
)

kept = sum(1 for idxs in by_speaker.values() if len(idxs) >= MIN_CLIPS_PER_SPK)
print(f"speakers with >= {MIN_CLIPS_PER_SPK} usable clips: {kept} / {len(by_speaker)}")
print(dataset)

speakers with >= 2 usable clips: 543 / 1474
Dataset({
    features: ['audio', 'text', 'source'],
    num_rows: 1423
})


In [ ]:
#@title Final dataset summary
import collections
counts = collections.Counter(dataset["source"])
pairable = sum(c for c in counts.values() if c >= 2)

print(dataset)
print(f"Rows: {len(dataset)} | speakers: {len(counts)} | clips per speaker: {len(dataset)/len(counts):.1f} avg")
print(f"Rows that can get a same-speaker reference prefix: {pairable} ({100*pairable/len(dataset):.1f}%)")

meta = dataset.select_columns(["text", "source"])
print("Example row:", meta[0])

Dataset({
    features: ['audio', 'text', 'source'],
    num_rows: 1423
})
Rows: 1423 | speakers: 543 | clips per speaker: 2.6 avg
Rows that can get a same-speaker reference prefix: 1423 (100.0%)
Example row: {'text': 'เออ งั้นก็ พี่ก็ดูแล้วกัน แล้วนางก็บอกว่า พี่ แล้ว เนี่ย', 'source': '7266'}


### ทำไมแท็ก `<laugh>`, `<sigh>`, `<gasp>` ฯลฯ ใช้ไม่ได้หลังเทรน

แท็กพวกนี้ **ไม่ใช่ token พิเศษ** ของ Orpheus — ใน `tokenizer_config.json` ไม่มีคำว่า `<laugh>` อยู่เลย
มันถูกตัดเป็นชิ้นข้อความธรรมดา (`<`, `laugh`, `>`) และ checkpoint `orpheus-3b-0.1-ft`
"เรียนรู้" จากข้อมูลเทรนว่าเจอชุดข้อความนี้แล้วให้เปล่งเสียงหัวเราะออกมา ไม่ใช่อ่านออกเสียง

ชุดข้อมูลไทย `yodas2_sidon_th_tts` เป็นข้อความถอดเสียงจาก ASR จึง **ไม่มีแท็กแม้แต่แถวเดียว**
เมื่อเทรน LoRA `r = 64` ครอบทุก linear layer ยาว 2,000 steps ที่ `lr = 2e-4`
พฤติกรรมเรื่องแท็กก็ถูกเขียนทับหายไปหมด (catastrophic forgetting) — นี่คือสาเหตุที่แท็กใช้ไม่ได้เลย

**สิ่งที่ *ไม่ควร* ทำ** คือ `tokenizer.add_special_tokens({"additional_special_tokens": [...]})`
ตามด้วย `model.resize_token_embeddings(...)` วิธีนี้ทำให้แท็กกลายเป็น token ใหม่ที่มี embedding
สุ่มขึ้นมา ความรู้เดิมของโมเดลถูกทิ้งทั้งหมด และต้องเทรนใหม่จากศูนย์

**สิ่งที่แก้จริงในโน้ตบุ๊กนี้** คือใส่ข้อมูลที่มีแท็กกลับเข้าไปในชุดเทรน (replay) 3 ชั้น:

| ขั้นตอน | ทำอะไร |
|---|---|
| ตรวจ tokenizer | ยืนยันว่าแท็กยังเป็นข้อความธรรมดา และ normalize รูปเขียน เช่น `<laughs>` → `<laugh>` |
| `mrfakename/Elise` | ผสมคลิปภาษาอังกฤษ ~3 ชม. ที่มีเสียงหัวเราะ/ถอนหายใจจริงพร้อมแท็ก (ใช้ทุกคลิปที่ใช้แท็กได้) |
| self-distillation | แท็กที่ Elise แทบไม่มี (`cough`, `sniffle`, `groan`, `yawn`, `gasp`) ให้ checkpoint ตั้งต้นสร้างตัวอย่างเสียงออกมาเอง แล้วป้อนกลับเข้าไปเทรน |

เสียงของแท็กไม่ขึ้นกับภาษา ดังนั้นการรักษาความเชื่อมโยง "แท็ก → เสียง" ไว้จากข้อมูลอังกฤษ
ก็เพียงพอให้สั่งแท็กในประโยคภาษาไทยได้ ถ้าอยากให้แท็กในบริบทไทยเนียนขึ้นอีก
ต้องหาคลิปเสียงไทยที่มีเสียงหัวเราะจริงมาติดแท็กเพิ่ม


In [ ]:
#@title Emotion tags: canonical list, tokeniser check and a normaliser
import re

# The eight tags the released orpheus-3b-0.1-ft checkpoint was trained to perform.
EMOTION_TAGS = ["<laugh>", "<chuckle>", "<sigh>", "<cough>", "<sniffle>", "<groan>", "<yawn>", "<gasp>"]

# These are NOT vocabulary entries - they are ordinary text that the checkpoint learned to
# render as a sound. Registering them with tokenizer.add_special_tokens() plus
# model.resize_token_embeddings() (a popular "fix" in the issue tracker) hands them fresh
# random embeddings and throws that pretrained behaviour away, so guard against it.
clashing = [t for t in EMOTION_TAGS if t in set(tokenizer.get_added_vocab())]
assert not clashing, (
    f"{clashing} were added to the vocabulary. Reload the tokeniser - these tags have to stay "
    "plain text or the pretrained laugh/sigh behaviour is lost."
)
for t in EMOTION_TAGS:
    print(f"{t:>10} -> {tokenizer.tokenize(t)}")

CANONICAL = {t.strip("<>") for t in EMOTION_TAGS}
TAG_ALIASES = { # what other datasets and humans write -> what Orpheus was trained on
    "laughs": "laugh", "laughing": "laugh", "laughter": "laugh",
    "laughs nervously": "laugh", "nervous laughter": "laugh",
    "chuckles": "chuckle", "chuckling": "chuckle",
    "giggle": "chuckle", "giggles": "chuckle", "giggling": "chuckle",
    "sighs": "sigh", "sighing": "sigh",
    "coughs": "cough", "coughing": "cough",
    "sniff": "sniffle", "sniffs": "sniffle", "sniffles": "sniffle", "sniffing": "sniffle",
    "groans": "groan", "groaning": "groan",
    "yawns": "yawn", "yawning": "yawn",
    "gasps": "gasp", "gasping": "gasp",
}
TAG_RE = re.compile(r"<\s*([A-Za-z][A-Za-z _'-]*?)\s*>")

def _canonical_name(raw):
    name = " ".join(raw.lower().split())
    return TAG_ALIASES.get(name, name)

def scan_tags(text):
    """-> (canonical tags found, raw tag strings we have no sound for)."""
    found, unknown = [], []
    for m in TAG_RE.finditer(text):
        name = _canonical_name(m.group(1))
        (found if name in CANONICAL else unknown).append(f"<{name}>" if name in CANONICAL else m.group(0))
    return found, unknown

def normalise_tags(text, drop_unknown = True):
    """Rewrite <laughs> as <laugh>, drop tags we cannot render, and fix the spacing.

    Spacing matters: a tag glued onto a neighbouring word tokenises differently from the
    stand-alone form the model saw during training, and then it does not get performed.
    """
    def sub(m):
        name = _canonical_name(m.group(1))
        if name in CANONICAL:
            return f" <{name}> "
        return " " if drop_unknown else m.group(0)
    text = TAG_RE.sub(sub, text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return re.sub(r"\s{2,}", " ", text).strip()


   <laugh> -> ['<', 'la', 'ugh', '>']
 <chuckle> -> ['<', 'ch', 'uckle', '>']
    <sigh> -> ['<s', 'igh', '>']
   <cough> -> ['<c', 'ough', '>']
 <sniffle> -> ['<', 'sn', 'iff', 'le', '>']
   <groan> -> ['<', 'gro', 'an', '>']
    <yawn> -> ['<y', 'awn', '>']
    <gasp> -> ['<g', 'asp', '>']


In [ ]:
#@title Hmong dataset removed
# Hmong dataset loading skipped as requested. Training continues with Thai/English dataset.
print(f"[✓] Current dataset ready: {len(dataset)} rows | {len(set(dataset['source']))} speakers")


Generating train split: 0 examples [00:00, ? examples/s]

/content/parquet_shards_wav2_tagged Dataset({
    features: ['audio', 'text', 'source', 'gender', 'speaker_id', 'speaker_name', 'emotion', 'emotion_confidence', 'has_vocal_tags'],
    num_rows: 2013
})
has_vocal_tags: {False: 1903, True: 110}
tags this checkpoint can perform: {'<gasp>': 74, '<laugh>': 30, '<chuckle>': 10}
tags with no Orpheus equivalent:   {}
keeping 2013 of 2013 rows
Dataset({
    features: ['audio', 'text', 'source'],
    num_rows: 3436
})
3436 rows | 551 speakers | 3436 rows can get a same-speaker reference


### เพิ่มเสียงอังกฤษแบบไม่มีแท็ก: `mythicinfinity/libritts_r`

LibriTTS-R คือ LibriTTS ที่ผ่านการ restore คุณภาพเสียงแล้ว (SLR141) เป็นชุดอ่านหนังสือหลายผู้พูด

| | |
|---|---|
| rows / hours | 33,232 คลิป, ~100 ชม. (split `train.clean.100` ทั้งก้อน) |
| speakers | 247 คน มี `speaker_id` จริง (~134 คลิปต่อคน) |
| sample rate | 24 kHz อยู่แล้ว — ตรงกับ SNAC ไม่ต้อง resample เลย ต่างจากชุดฮ์มอง 16 kHz |
| text | `text_normalized` ตัวเลข/ตัวย่อถูกกระจายเป็นคำอ่านแล้ว |
| license | CC BY 4.0 ไม่ gated ไม่ต้องขอสิทธิ์ |

ชุดนี้ **ไม่มีแท็กอารมณ์** ใส่เข้ามาเพื่อให้โมเดลพูดอังกฤษได้ ไม่ได้ช่วยเรื่องแท็ก —
เรื่องแท็กยังเป็นหน้าที่ของ cell `mrfakename/Elise` และ self-distillation ที่อยู่ถัดไป

แถวที่ผ่านตัวกรองความยาวข้อความจะถูก **รวมทั้งหมด** เข้า `dataset` ไม่มีการตัดสัดส่วน
ถ้าดิสก์ Colab แน่น ให้ลด `EN_N_SHARDS` (ค่าเริ่มต้น = 18 = ทั้ง `train.clean.100`)


In [ ]:
#@title Add untagged English speech: LibriTTS-R (native 24 kHz)
from datasets import load_dataset, Audio, Value, concatenate_datasets
import collections

EN_REPO  = "mythicinfinity/libritts_r"
EN_SPLIT = "train.clean.100"
EN_TOTAL_SHARDS = 18

# Rows are grouped by speaker. Raise EN_N_SHARDS to pull more English in; everything
# that survives the length filter is concatenated - no ratio trim.
#   1 shard  ->  1,847 rows,  15 speakers, ~0.5 GB
#   5 shards ->  9,235 rows,  75 speakers, ~2.6 GB
#   8 shards -> 14,776 rows, 120 speakers, ~4.1 GB
#  18 shards -> 33,232 rows, 247 speakers, ~10 GB  (full train.clean.100)
EN_N_SHARDS = 1
EN_MIN_CHARS, EN_MAX_CHARS = 25, 180

shards = [f"data/{EN_SPLIT}/{EN_SPLIT}-{i:05d}-of-{EN_TOTAL_SHARDS:05d}.parquet"
          for i in range(EN_N_SHARDS)]
raw_en = load_dataset(EN_REPO, data_files = {"train": shards}, split = "train")

# The parquet footer already declares Audio(sampling_rate = 24000); this only makes it explicit,
# and unlike the Hmong set nothing is resampled, so these rows keep their full band.
raw_en = raw_en.cast_column("audio", Audio(sampling_rate = 24000))
print(raw_en)

texts    = raw_en["text_normalized"]  # reading single columns decodes no audio
speakers = raw_en["speaker_id"]

# Check the premise instead of trusting it: read speech should carry no tag-like markup, so
# nothing here should need normalise_tags() or get dropped the way the Elise rows do.
stray = collections.Counter()
for t in texts:
    found, unknown = scan_tags(t)
    stray.update(found + unknown)
print("tag-like markup in LibriTTS:", dict(stray) or "none, as expected")

# No duration column to filter on, so text length stands in for it: English reads at roughly
# 15 characters a second, putting 25-180 chars in the same 1.5-12 s window as MIN/MAX_DUR.
keep = [i for i, t in enumerate(texts) if EN_MIN_CHARS <= len(t) <= EN_MAX_CHARS]
print(f"keeping {len(keep)} of {len(texts)} English rows (length filter only)")

# The "en_" prefix stops create_input_ids() from handing a Thai or Hmong target an English
# reference voice, the same reason the Hmong ids are prefixed.
ds_en = (
    raw_en.select(keep)
          .flatten_indices()
          .remove_columns([c for c in raw_en.column_names if c != "audio"])
          .add_column("text", [texts[i] for i in keep])
          .add_column("source", [f"en_{speakers[i]}" for i in keep])
)

if dataset.features["source"] != Value("string"):
    dataset = dataset.cast_column("source", Value("string"))
dataset = concatenate_datasets([dataset, ds_en]).shuffle(seed = 3407)

counts = collections.Counter(dataset["source"])
print(dataset)
print(f"{len(dataset)} rows | {len(counts)} speakers | "
      f"English is {100 * len(ds_en) / len(dataset):.1f}% of the mix")


Dataset({
    features: ['audio', 'text_normalized', 'text_original', 'speaker_id', 'path', 'chapter_id', 'id'],
    num_rows: 1847
})
tag-like markup in LibriTTS: none, as expected
keeping 1415 of 1847 English rows (length filter only)
Dataset({
    features: ['audio', 'text', 'source'],
    num_rows: 6266
})
6266 rows | 566 speakers | English is 22.6% of the mix


In [ ]:
#@title Mix in clips whose audio really contains the tagged sounds
from datasets import load_dataset, Audio, concatenate_datasets
import collections

TAG_REPLAY_REPO = "mrfakename/Elise" # public MIT clone of the Elise emotive set, ~3 h English

replay = load_dataset(TAG_REPLAY_REPO, split = "train")
replay = replay.cast_column("audio", Audio(sampling_rate = 24000))

# Only the text column is touched here so that no audio gets decoded during the scan.
keep, dropped, per_tag, clean_text = [], 0, collections.Counter(), {}
for i, raw_text in enumerate(replay["text"]):
    found, unknown = scan_tags(raw_text)
    if unknown:
        # e.g. <whispers>, <singing>, <clears throat>: the sound is in the audio but we have
        # no tag for it, and an untagged sound teaches the model to insert it uninvited.
        dropped += 1
        continue
    if not found:
        continue
    keep.append(i)
    per_tag.update(found)
    clean_text[i] = normalise_tags(raw_text)

print(f"{len(keep)} usable tagged clips, {dropped} dropped for carrying an unsupported tag")
print("occurrences per tag in the source:", dict(per_tag))

if keep:
    # Keep every usable tagged clip once - no ratio trim, no oversampling.
    replay = (
        replay.select(keep)
              .flatten_indices()
              .remove_columns("text")
              .add_column("text", [clean_text[i] for i in keep])
              .add_column("source", ["tag_replay_en"] * len(keep))
              .select_columns(["audio", "text", "source"])
    )
    dataset = concatenate_datasets([dataset, replay]).shuffle(seed = 3407)
    print(f"tag-replay rows: {len(replay)} ({100 * len(replay) / len(dataset):.1f}% of {len(dataset)})")
else:
    print("No tag-replay rows mixed in - expect the pretrained tag behaviour to be trained away.")

# Final mix summary before SNAC tokenisation.
_groups = collections.Counter()
for s in dataset["source"]:
    if s == "tag_replay_en":
        _groups["elise/tag"] += 1
    elif s.startswith("hmn2_"):
        _groups["hmong2"] += 1
    elif s.startswith("hmn_"):
        _groups["hmong"] += 1
    elif s.startswith("en_"):
        _groups["english"] += 1
    else:
        _groups["thai"] += 1
print("combined training mix:", dict(_groups), "| total:", len(dataset))


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/328M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1195 [00:00<?, ? examples/s]

284 usable tagged clips, 43 dropped for carrying an unsupported tag
occurrences per tag in the source: {'<laugh>': 171, '<chuckle>': 47, '<sigh>': 81, '<sniffle>': 3, '<yawn>': 1, '<gasp>': 2, '<cough>': 1}


Flattening the indices:   0%|          | 0/284 [00:00<?, ? examples/s]

tag-replay rows: 284 (7.6% of 3720)
combined training mix: {'hmong': 2013, 'thai': 1423, 'elise/tag': 284} | total: 3720


In [ ]:
#@title Tokenization Function: encode every clip into SNAC audio tokens

import locale
import torchaudio.transforms as T
import os
import torch
from snac import SNAC
locale.getpreferredencoding = lambda: "UTF-8"

snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz")
snac_model = snac_model.to("cuda")

# Seven SNAC tokens make up one frame, and the coarsest layer runs at 24000/2048 Hz
SNAC_FRAME_RATE = 24000 / 2048
_resamplers = {}

def tokenise_audio(waveform, sample_rate = 24000):
  # The rate is read per row now: the Thai clips and the tag-replay clips are concatenated,
  # so one global rate sampled from row 0 would silently resample the other set wrongly.
  waveform = torch.from_numpy(waveform).unsqueeze(0)
  waveform = waveform.to(dtype = torch.float32)
  if sample_rate != 24000:
    if sample_rate not in _resamplers:
      _resamplers[sample_rate] = T.Resample(orig_freq = sample_rate, new_freq = 24000)
    waveform = _resamplers[sample_rate](waveform)

  waveform = waveform.unsqueeze(0).to("cuda")

  #generate the codes from snac
  with torch.inference_mode():
    codes = snac_model.encode(waveform)

  all_codes = []
  for i in range(codes[0].shape[1]):
    all_codes.append(codes[0][0][i].item()+128266)
    all_codes.append(codes[1][0][2*i].item()+128266+4096)
    all_codes.append(codes[2][0][4*i].item()+128266+(2*4096))
    all_codes.append(codes[2][0][(4*i)+1].item()+128266+(3*4096))
    all_codes.append(codes[1][0][(2*i)+1].item()+128266+(4*4096))
    all_codes.append(codes[2][0][(4*i)+2].item()+128266+(5*4096))
    all_codes.append(codes[2][0][(4*i)+3].item()+128266+(6*4096))


  return all_codes

def add_codes(example):
    # Always initialize codes_list to None
    codes_list = None

    try:
        answer_audio = example.get("audio")
        # If there's a valid audio array, tokenise it
        if answer_audio and "array" in answer_audio:
            codes_list = tokenise_audio(answer_audio["array"],
                                        answer_audio.get("sampling_rate", 24000))
    except Exception as e:
        print(f"Skipping row due to error: {e}")
        # Keep codes_list as None if we fail
    example["codes_list"] = codes_list

    return example

dataset = dataset.map(add_codes, remove_columns = ["audio"])

tokeniser_length = 128256
start_of_text = 128000
end_of_text = 128009

start_of_speech = tokeniser_length + 1
end_of_speech = tokeniser_length + 2

start_of_human = tokeniser_length + 3
end_of_human = tokeniser_length + 4

start_of_ai = tokeniser_length + 5
end_of_ai =  tokeniser_length + 6
pad_token = tokeniser_length + 7

audio_tokens_start = tokeniser_length + 10

dataset = dataset.filter(lambda x: x["codes_list"] is not None)
dataset = dataset.filter(lambda x: len(x["codes_list"]) > 0)

def remove_duplicate_frames(example):
    vals = example["codes_list"]
    if len(vals) % 7 != 0:
        raise ValueError("Input list length must be divisible by 7")

    result = vals[:7]

    removed_frames = 0

    for i in range(7, len(vals), 7):
        current_first = vals[i]
        previous_first = result[-7]

        if current_first != previous_first:
            result.extend(vals[i:i+7])
        else:
            removed_frames += 1

    example["codes_list"] = result

    return example

dataset = dataset.map(remove_duplicate_frames)

def encode_waveform_to_codes(waveform, sr, device = None):
    """waveform: 1D tensor/ndarray mono. Returns list of SNAC token ids with offsets applied."""
    device = device or next(snac_model.parameters()).device
    if not torch.is_tensor(waveform):
        waveform = torch.from_numpy(waveform)
    waveform = waveform.to(torch.float32).reshape(1, -1)
    if sr != 24000:
        waveform = T.Resample(orig_freq = sr, new_freq = 24000)(waveform)
    waveform = waveform.unsqueeze(0).to(device)
    with torch.inference_mode():
        codes = snac_model.encode(waveform)
    all_codes = []
    for i in range(codes[0].shape[1]):
        all_codes.append(codes[0][0][i].item()+128266)
        all_codes.append(codes[1][0][2*i].item()+128266+4096)
        all_codes.append(codes[2][0][4*i].item()+128266+(2*4096))
        all_codes.append(codes[2][0][(4*i)+1].item()+128266+(3*4096))
        all_codes.append(codes[1][0][(2*i)+1].item()+128266+(4*4096))
        all_codes.append(codes[2][0][(4*i)+2].item()+128266+(5*4096))
        all_codes.append(codes[2][0][(4*i)+3].item()+128266+(6*4096))
    return all_codes

def load_ref_codes(path, max_seconds = 6.0):
    """Load a reference audio file and return de-duplicated SNAC codes to use as a voice prompt."""
    import torchaudio
    wav, sr = torchaudio.load(path)
    wav = wav.mean(dim = 0)                        # force mono
    wav = wav[: int(sr * max_seconds)]              # cap length so the prefix doesn't blow the context
    codes = encode_waveform_to_codes(wav, sr)
    return remove_duplicate_frames({"codes_list": codes})["codes_list"]

print(dataset)
print(f"Clips tokenised into SNAC codes: {len(dataset)}")

config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/79.5M [00:00<?, ?B/s]

Parameter 'function'=<function add_codes at 0x7bbf135ffce0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/3720 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3720 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3720 [00:00<?, ? examples/s]

Map:   0%|          | 0/3720 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'source', 'codes_list'],
    num_rows: 3720
})
Clips tokenised into SNAC codes: 3720


In [ ]:
def load_ref_codes(path, max_seconds = 6.0):
    """Load a reference audio file and return de-duplicated SNAC codes to use as a voice prompt."""
    import torchaudio
    wav, sr = torchaudio.load(path)
    wav = wav.mean(dim = 0)                        # force mono
    wav = wav[: int(sr * max_seconds)]              # cap length so the prefix doesn't blow the context
    codes = encode_waveform_to_codes(wav, sr)
    return remove_duplicate_frames({"codes_list": codes})["codes_list"]


In [ ]:
#@title Generation helpers (shared by the tag distillation and the tag check further down)

TOK_SOH, TOK_EOT, TOK_EOH = 128259, 128009, 128260
TOK_SOS, TOK_EOS, TOK_PAD = 128257, 128258, 128263
AUDIO_TOKEN_START = 128266

def _left_pad(seqs):
    width = max(s.shape[1] for s in seqs)
    ids, mask = [], []
    for s in seqs:
        pad = width - s.shape[1]
        ids.append(torch.cat([torch.full((1, pad), TOK_PAD, dtype = torch.int64), s], dim = 1))
        mask.append(torch.cat([torch.zeros((1, pad), dtype = torch.int64),
                               torch.ones((1, s.shape[1]), dtype = torch.int64)], dim = 1))
    return torch.cat(ids, 0), torch.cat(mask, 0)

def extract_codes(row):
    """One generated sequence -> (zero-based SNAC ids trimmed to whole frames, ended cleanly)."""
    hits = (row == TOK_SOS).nonzero(as_tuple = True)[0]
    row = row[hits[-1].item() + 1:] if len(hits) else row
    stop = (row == TOK_EOS).nonzero(as_tuple = True)[0]
    finished = len(stop) > 0
    if finished:
        row = row[: stop[0].item()]
    row = row[row >= AUDIO_TOKEN_START]
    row = row[: (row.size(0) // 7) * 7]
    return [t.item() - AUDIO_TOKEN_START for t in row], finished

def codes_are_valid(code_list):
    """Each slot in a 7-token frame owns its own 4096-wide band; anything else decodes to noise."""
    if not code_list or len(code_list) % 7:
        return False
    return all(0 <= v - (j % 7) * 4096 < 4096 for j, v in enumerate(code_list))

def orpheus_generate_codes(texts, voice = None, use_base = False, batch_size = 8,
                           temperature = 0.6, top_p = 0.95, repetition_penalty = 1.1,
                           max_new_tokens = 1200):
    """Generate SNAC codes for a list of texts. use_base = True bypasses the LoRA adapter."""
    FastLanguageModel.for_inference(model)

    def run():
        results = []
        for start in range(0, len(texts), batch_size):
            seqs = []
            for t in texts[start : start + batch_size]:
                body = tokenizer(f"{voice}: {t}" if voice else t, return_tensors = "pt").input_ids
                seqs.append(torch.cat([
                    torch.tensor([[TOK_SOH]], dtype = torch.int64),
                    body,
                    torch.tensor([[TOK_EOT, TOK_EOH]], dtype = torch.int64),
                ], dim = 1))
            ids, mask = _left_pad(seqs)
            with torch.inference_mode():
                generated = model.generate(
                    input_ids = ids.to("cuda"),
                    attention_mask = mask.to("cuda"),
                    max_new_tokens = max_new_tokens,
                    do_sample = True,
                    temperature = temperature,
                    top_p = top_p,
                    # Orpheus needs this above 1.0 or it stalls on a repeating frame
                    repetition_penalty = repetition_penalty,
                    eos_token_id = TOK_EOS,
                    use_cache = True,
                )
            results.extend(extract_codes(r) for r in generated)
        return results

    if use_base and hasattr(model, "disable_adapter"):
        with model.disable_adapter():
            return run()
    if use_base:
        print("No disable_adapter() on this model - generating with the adapter attached. "
              "Before training that is harmless: an untrained LoRA is still an identity map.")
    return run()

def codes_to_audio(code_list):
    """Undo the 7-token interleave, rebuild the three SNAC layers and decode to a waveform."""
    layer_1, layer_2, layer_3 = [], [], []
    for i in range(len(code_list) // 7):
        f = code_list[7 * i : 7 * i + 7]
        layer_1.append(f[0])
        layer_2.append(f[1] - 4096)
        layer_3.append(f[2] - 2 * 4096)
        layer_3.append(f[3] - 3 * 4096)
        layer_2.append(f[4] - 4 * 4096)
        layer_3.append(f[5] - 5 * 4096)
        layer_3.append(f[6] - 6 * 4096)
    device = next(snac_model.parameters()).device
    codes = [torch.tensor(l, dtype = torch.int64).unsqueeze(0).to(device)
             for l in (layer_1, layer_2, layer_3)]
    with torch.inference_mode():
        return snac_model.decode(codes).squeeze().detach().cpu().numpy()


In [ ]:
#@title Top up the rare tags by distilling them out of the base checkpoint

# Elise covers laugh / chuckle / sigh well but has almost nothing for cough, sniffle,
# groan, yawn or gasp. The released checkpoint already performs all eight, so we ask it
# for the missing ones and feed its own output back in as replay data. This preserves a
# skill rather than teaching a new one, which is exactly what we need here.
# RUN THIS BEFORE trainer.train() - freshly initialised LoRA weights are still an identity
# map, so the samples come from the untouched base behaviour.

SELF_DISTIL_TAGS = True   # set False to skip (costs roughly 5-10 min on a T4)
DISTIL_PER_TAG   = 16     # only tags with fewer occurrences than this get generated
DISTIL_VOICE     = "tara" # the released checkpoint performs tags best behind its own voices

CARRIERS = {
    "<laugh>": [
        "That is the funniest thing I have heard all week. <laugh>",
        "You actually fell for it? <laugh> I cannot believe that worked.",
        "<laugh> Oh no, do not make me start again.",
        "And then the cat knocked the whole cup over. <laugh>",
        "I told you it was a terrible idea. <laugh>",
        "Wait, you wore that to the interview? <laugh>",
    ],
    "<chuckle>": [
        "Well, that is one way to do it. <chuckle>",
        "<chuckle> You are impossible sometimes.",
        "I suppose that makes sense. <chuckle>",
        "You always say that. <chuckle> Every single time.",
        "Fair enough. <chuckle> I will let it slide.",
        "That is a bit cheeky. <chuckle>",
    ],
    "<sigh>": [
        "<sigh> Fine, I will do it myself.",
        "It has been a very long day. <sigh>",
        "<sigh> I really thought this would be easier.",
        "Another form to fill in. <sigh>",
        "I miss having free weekends. <sigh>",
        "<sigh> Let us just start over from the beginning.",
    ],
    "<cough>": [
        "Excuse me. <cough> Sorry, my throat is dry.",
        "<cough> Sorry about that, where were we?",
        "The dust in here is terrible. <cough>",
        "<cough> I think I am coming down with something.",
        "Could you pass me the water? <cough>",
        "It is so smoky in this room. <cough>",
    ],
    "<sniffle>": [
        "<sniffle> Sorry, my allergies are awful today.",
        "I forgot my tissues again. <sniffle>",
        "<sniffle> It is freezing out there.",
        "I have had this cold all week. <sniffle>",
        "<sniffle> Do not mind me, I am fine.",
        "The pollen count must be sky high. <sniffle>",
    ],
    "<groan>": [
        "<groan> Not another meeting about meetings.",
        "The server is down again. <groan>",
        "<groan> I cannot look at this spreadsheet any longer.",
        "You are telling me we have to start over? <groan>",
        "<groan> My back hurts from sitting all day.",
        "It is only Tuesday. <groan>",
    ],
    "<yawn>": [
        "<yawn> Sorry, I barely slept last night.",
        "I think I need another coffee. <yawn>",
        "<yawn> What time is it anyway?",
        "It is way past my bedtime. <yawn>",
        "<yawn> Let us pick this up in the morning.",
        "Long flights always wear me out. <yawn>",
    ],
    "<gasp>": [
        "<gasp> You are joking, right?",
        "Wait, she said what? <gasp>",
        "<gasp> I completely forgot about the deadline.",
        "Look at that view. <gasp>",
        "<gasp> Do not sneak up on me like that.",
        "They cancelled the whole thing? <gasp>",
    ],
}

if SELF_DISTIL_TAGS:
    import collections
    from datasets import Dataset, concatenate_datasets

    have = collections.Counter()
    for t in dataset["text"]:
        have.update(scan_tags(t)[0])

    wanted = []
    for tag in EMOTION_TAGS:
        shortfall = DISTIL_PER_TAG - have[tag]
        carriers = CARRIERS[tag]
        wanted += [carriers[i % len(carriers)] for i in range(max(0, shortfall))]
    print(f"Generating {len(wanted)} clips for {sorted({t for t in EMOTION_TAGS if have[t] < DISTIL_PER_TAG})}")

    rows = {"text": [], "source": [], "codes_list": []}
    if wanted:
        for text, (codes, finished) in zip(wanted, orpheus_generate_codes(
                wanted, voice = DISTIL_VOICE, use_base = True, max_new_tokens = 500)):
            seconds = (len(codes) // 7) / SNAC_FRAME_RATE
            # A run that never emitted end-of-speech was cut off mid-word, and codes that
            # land outside their layer's range would decode into noise.
            if not finished or not codes_are_valid(codes) or not 1.0 <= seconds <= 14.0:
                continue
            rows["text"].append(text)                       # trained without the voice prefix
            rows["source"].append("tag_distil_en")
            rows["codes_list"].append(remove_duplicate_frames({"codes_list": codes})["codes_list"])
        print(f"Kept {len(rows['text'])} of {len(wanted)} generations")

    if rows["text"]:
        extra = Dataset.from_dict(rows, features = dataset.features)
        dataset = concatenate_datasets([dataset, extra]).shuffle(seed = 3407)

    FastLanguageModel.for_training(model) # undo for_inference() before the Trainer runs

print(dataset)


Generating 59 clips for ['<cough>', '<groan>', '<sniffle>', '<yawn>']
Kept 32 of 59 generations
Dataset({
    features: ['text', 'source', 'codes_list'],
    num_rows: 3752
})


In [ ]:
#@title Build the training sequences

tok_info = '''*** HERE you can modify the text prompt
This trains for zero-shot voice cloning, so the text stays plain - no
"speaker_name: " tag. The speaker ids here are arbitrary YouTube-derived labels that
each appear only a handful of times, so tagging them into the text would just be noise,
and at inference time you would not have an id to ask for anyway. Voice identity is
carried entirely by the reference-audio turn instead.

REF_PROB of the examples get a *completed* utterance (text + audio codes) from the SAME
speaker prepended. The loss is masked over that prefix, so the model only learns to
produce the target turn while conditioning on the reference voice. The remaining
examples train plain TTS, which keeps the model usable without a reference clip.

Emotion tags stay inline in the text exactly as they appear in the prompt at inference
time - they are ordinary text, so nothing special happens to them here.
'''
print(tok_info)

import collections, random
random.seed(3407)

tag_counts = collections.Counter()
for t in dataset["text"]:
    tag_counts.update(scan_tags(t)[0])
print("tag occurrences going into training:", {t: tag_counts[t] for t in EMOTION_TAGS})
missing = [t for t in EMOTION_TAGS if tag_counts[t] == 0]
if missing:
    print(f"WARNING: {missing} have no training examples and will not work at inference.")

speaker_to_indices = {}
for i, spk in enumerate(dataset["source"]):
    speaker_to_indices.setdefault(spk, []).append(i)

REF_PROB = 0.8      # probability that a training example gets a reference-audio prefix
MAX_REF_FRAMES = 96 # cap the reference turn at ~8s of audio (7 tokens per SNAC frame)
MAX_LEN = max_seq_length

def build_turn(text, codes):
    ids = tokenizer.encode(text, add_special_tokens = True) + [end_of_text]
    return [start_of_human] + ids + [end_of_human, start_of_ai, start_of_speech] + codes + [end_of_speech, end_of_ai]

def create_input_ids(example, idx):
    spk = example["source"]
    tgt_codes = example["codes_list"]

    prefix, prefix_len = [], 0
    pool = [j for j in speaker_to_indices[spk] if j != idx]
    if pool and random.random() < REF_PROB:
        ref = dataset[random.choice(pool)]
        prefix = build_turn(ref["text"], ref["codes_list"][: MAX_REF_FRAMES * 7])
        prefix_len = len(prefix)

    body = build_turn(example["text"], tgt_codes)
    input_ids = prefix + body
    # only train on the target turn; mask the reference-audio prefix out of the loss
    labels = [-100] * prefix_len + body

    example["input_ids"] = input_ids
    example["labels"] = labels
    example["attention_mask"] = [1] * len(input_ids)

    return example


dataset = dataset.map(create_input_ids, with_indices = True, remove_columns = ["text", "codes_list"])

before = len(dataset)
dataset = dataset.filter(lambda x: len(x["input_ids"]) <= MAX_LEN)
print(f"Dropped {before - len(dataset)} of {before} rows for exceeding MAX_LEN = {MAX_LEN}")

columns_to_keep = ["input_ids", "labels", "attention_mask"]
columns_to_remove = [col for col in dataset.column_names if col not in columns_to_keep]

dataset = dataset.remove_columns(columns_to_remove)

# Sanity check: a masked prefix means the example actually carries a reference voice.
# If this reads 0%, the dataset has no speaker with two clips and nothing zero-shot is
# being learned - which is exactly the failure mode this pipeline is set up to avoid.
sample = dataset.select(range(min(500, len(dataset))))
with_ref = sum(1 for labels in sample["labels"] if labels[0] == -100)
print(f"Examples carrying a reference-audio prefix: {with_ref}/{len(sample)} ({100*with_ref/len(sample):.0f}%)")


*** HERE you can modify the text prompt
This trains for zero-shot voice cloning, so the text stays plain - no
"speaker_name: " tag. The speaker ids here are arbitrary YouTube-derived labels that
each appear only a handful of times, so tagging them into the text would just be noise,
and at inference time you would not have an id to ask for anyway. Voice identity is
carried entirely by the reference-audio turn instead.

REF_PROB of the examples get a *completed* utterance (text + audio codes) from the SAME
speaker prepended. The loss is masked over that prefix, so the model only learns to
produce the target turn while conditioning on the reference voice. The remaining
examples train plain TTS, which keeps the model usable without a reference clip.

Emotion tags stay inline in the text exactly as they appear in the prompt at inference
time - they are ordinary text, so nothing special happens to them here.

tag occurrences going into training: {'<laugh>': 201, '<chuckle>': 57, '<sigh>': 81

Map:   0%|          | 0/3752 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3752 [00:00<?, ? examples/s]

Dropped 0 of 3752 rows for exceeding MAX_LEN = 4096
Examples carrying a reference-audio prefix: 401/500 (80%)


<a name="Train"></a>
### Train the model
Now let's use Hugging Face `Trainer`! More docs here: [Transformers docs](https://huggingface.co/docs/transformers/main_classes/trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

**Note:** Using a per_device_train_batch_size >1 may lead to errors if multi-GPU setup to avoid issues, ensure CUDA_VISIBLE_DEVICES is set to a single GPU (e.g., CUDA_VISIBLE_DEVICES=0).

In [ ]:
from transformers import TrainingArguments,Trainer,DataCollatorForSeq2Seq
trainer = Trainer(
    model = model,
    train_dataset = dataset,
    args = TrainingArguments(
        # Keep batch size at 1: no data collator is passed, so variable-length rows
        # cannot be batched together without padding.
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,
        # 60 steps only ever showed the model ~240 clips, nowhere near enough to pick up
        # Thai and reference-conditioning. Set num_train_epochs = 1 and max_steps = -1
        # for a full pass over the data.
        max_steps = 8000,
        learning_rate = 2e-4,
        logging_steps = 100,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.034 GB.
7.111 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint=True)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,752 | Num Epochs = 9 | Total steps = 8,000
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 24,313,856 of 3,325,180,928 (0.73% trained)


Step,Training Loss


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

NameError: name 'start_gpu_memory' is not defined

<a name="Inference"></a>
### Inference
Let's run the model! You can change the prompts

In [ ]:
prompts = [
    # "สวัสดีครับ ผมชื่อปกรณ์",
    "วันนี้เป็นวันที่ดีมากมากเลย สวัสดีครับผม",
    "ib os peb",
]

# Fixes spacing and rewrites near-misses like <laughs> or <giggle> into the eight tags the
# model was actually trained on. A tag glued to a word does not get performed reliably.
prompts = [normalise_tags(p) for p in prompts]
for p in prompts:
    print(scan_tags(p)[0], p)

chosen_voice = None # Must stay None: training used plain text with no speaker tag

# Zero-shot cloning: point this at a few seconds of the voice you want to copy.
# Leave it None to just hear the fine-tuned default voice.
ref_audio_path = "/content/download.wav" # e.g. "ref.wav"
ref_text = "Zoo siab tau ntsib nej txhua tus."         # transcript that matches ref_audio_path word-for-word

[] วันนี้เป็นวันที่ดีมากมากเลย สวัสดีครับผม
[] Ib Os Peb


In [ ]:
#@title Run Inference


# Emotion tags have to reach the model as the exact plain-text strings it was trained on,
# so refuse anything the tokeniser would turn into a different sequence of pieces.
for p in prompts:
    _, unknown = scan_tags(p)
    if unknown:
        raise ValueError(
            f"{unknown} are not tags this model knows. Run normalise_tags() on the prompt, "
            f"or stick to {EMOTION_TAGS}."
        )

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Encode the reference audio (if any) while SNAC is still on GPU - much faster than on CPU
ref_codes = load_ref_codes(ref_audio_path) if ref_audio_path else []

# Moving snac_model cuda to cpu
snac_model.to("cpu")

prompts_ = [(f"{chosen_voice}: " + p) if chosen_voice else p for p in prompts]

all_input_ids = []

for prompt in prompts_:
  input_ids = tokenizer(prompt, return_tensors = "pt").input_ids
  all_input_ids.append(input_ids)

start_token = torch.tensor([[ 128259]], dtype = torch.int64) # Start of human
end_tokens = torch.tensor([[128009, 128260]], dtype = torch.int64) # End of text, End of human

# Build a completed reference turn to prepend, so the model continues in that same voice
ref_prefix = []
if ref_codes:
  ref_prompt = f"{chosen_voice}: {ref_text}" if chosen_voice else ref_text
  ref_text_ids = tokenizer.encode(ref_prompt, add_special_tokens = True) + [128009] # End of text
  ref_prefix = [128259] + ref_text_ids + [128260, 128261, 128257] + ref_codes + [128258, 128262] # SOH..EOH SOAI SOS codes EOS EOAI
ref_prefix = torch.tensor([ref_prefix], dtype = torch.int64) if ref_prefix else torch.zeros((1, 0), dtype = torch.int64)

all_modified_input_ids = []
for input_ids in all_input_ids:
  modified_input_ids = torch.cat([ref_prefix, start_token, input_ids, end_tokens], dim = 1) # [ref turn] SOH SOT Text EOT EOH
  all_modified_input_ids.append(modified_input_ids)

all_padded_tensors = []
all_attention_masks = []
max_length = max([modified_input_ids.shape[1] for modified_input_ids in all_modified_input_ids])
for modified_input_ids in all_modified_input_ids:
  padding = max_length - modified_input_ids.shape[1]
  padded_tensor = torch.cat([torch.full((1, padding), 128263, dtype = torch.int64), modified_input_ids], dim = 1)
  attention_mask = torch.cat([torch.zeros((1, padding), dtype = torch.int64), torch.ones((1, modified_input_ids.shape[1]), dtype = torch.int64)], dim = 1)
  all_padded_tensors.append(padded_tensor)
  all_attention_masks.append(attention_mask)

all_padded_tensors = torch.cat(all_padded_tensors, dim = 0)
all_attention_masks = torch.cat(all_attention_masks, dim = 0)

input_ids = all_padded_tensors.to("cuda")
attention_mask = all_attention_masks.to("cuda")
generated_ids = model.generate(
      input_ids = input_ids,
      attention_mask = attention_mask,
      max_new_tokens = 1200,
      do_sample = True,
      temperature = 0.6,
      top_p = 0.95,
      repetition_penalty = 1.1,
      num_return_sequences = 1,
      eos_token_id = 128258,
     use_cache = True
  )
token_to_find = 128257   # Start of speech
token_to_remove = 128258 # End of speech

# Crop per-row (not per-batch): when a reference-audio prefix is used, its own
# Start-of-speech token would otherwise throw off a single batch-wide crop index.
code_lists = []
for row in generated_ids:
    hits = (row == token_to_find).nonzero(as_tuple = True)[0]
    row = row[hits[-1].item() + 1:] if len(hits) > 0 else row
    stop = (row == token_to_remove).nonzero(as_tuple = True)[0]
    if len(stop) > 0:
        row = row[: stop[0].item()]
    row = row[row >= 128266] # drop any stray special/pad tokens
    row_length = row.size(0)
    new_length = (row_length // 7) * 7
    trimmed_row = row[:new_length]
    trimmed_row = [t.item() - 128266 for t in trimmed_row]
    code_lists.append(trimmed_row)


def redistribute_codes(code_list):
  layer_1 = []
  layer_2 = []
  layer_3 = []
  for i in range((len(code_list)+1)//7):
    layer_1.append(code_list[7*i])
    layer_2.append(code_list[7*i+1]-4096)
    layer_3.append(code_list[7*i+2]-(2*4096))
    layer_3.append(code_list[7*i+3]-(3*4096))
    layer_2.append(code_list[7*i+4]-(4*4096))
    layer_3.append(code_list[7*i+5]-(5*4096))
    layer_3.append(code_list[7*i+6]-(6*4096))
  codes = [torch.tensor(layer_1).unsqueeze(0),
         torch.tensor(layer_2).unsqueeze(0),
         torch.tensor(layer_3).unsqueeze(0)]

  # codes = [c.to("cuda") for c in codes]
  audio_hat = snac_model.decode(codes)
  return audio_hat

my_samples = []
for code_list in code_lists:
  samples = redistribute_codes(code_list)
  my_samples.append(samples)
from IPython.display import display, Audio
if len(prompts) != len(my_samples):
  raise Exception("Number of prompts and samples do not match")
else:
  for i in range(len(my_samples)):
    print(prompts[i])
    samples = my_samples[i]
    display(Audio(samples.detach().squeeze().to("cpu").numpy(), rate = 24000))
# Clean up to save RAM
del my_samples,samples

วันนี้เป็นวันที่ดีมากมากเลย สวัสดีครับผม


ib os peb


In [ ]:
#@title Run Inference


# Emotion tags have to reach the model as the exact plain-text strings it was trained on,
# so refuse anything the tokeniser would turn into a different sequence of pieces.
for p in prompts:
    _, unknown = scan_tags(p)
    if unknown:
        raise ValueError(
            f"{unknown} are not tags this model knows. Run normalise_tags() on the prompt, "
            f"or stick to {EMOTION_TAGS}."
        )

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Encode the reference audio (if any) while SNAC is still on GPU - much faster than on CPU
ref_codes = load_ref_codes(ref_audio_path) if ref_audio_path else []

# Moving snac_model cuda to cpu
snac_model.to("cpu")

prompts_ = [(f"{chosen_voice}: " + p) if chosen_voice else p for p in prompts]

all_input_ids = []

for prompt in prompts_:
  input_ids = tokenizer(prompt, return_tensors = "pt").input_ids
  all_input_ids.append(input_ids)

start_token = torch.tensor([[ 128259]], dtype = torch.int64) # Start of human
end_tokens = torch.tensor([[128009, 128260]], dtype = torch.int64) # End of text, End of human

# Build a completed reference turn to prepend, so the model continues in that same voice
ref_prefix = []
if ref_codes:
  ref_prompt = f"{chosen_voice}: {ref_text}" if chosen_voice else ref_text
  ref_text_ids = tokenizer.encode(ref_prompt, add_special_tokens = True) + [128009] # End of text
  ref_prefix = [128259] + ref_text_ids + [128260, 128261, 128257] + ref_codes + [128258, 128262] # SOH..EOH SOAI SOS codes EOS EOAI
ref_prefix = torch.tensor([ref_prefix], dtype = torch.int64) if ref_prefix else torch.zeros((1, 0), dtype = torch.int64)

all_modified_input_ids = []
for input_ids in all_input_ids:
  modified_input_ids = torch.cat([ref_prefix, start_token, input_ids, end_tokens], dim = 1) # [ref turn] SOH SOT Text EOT EOH
  all_modified_input_ids.append(modified_input_ids)

all_padded_tensors = []
all_attention_masks = []
max_length = max([modified_input_ids.shape[1] for modified_input_ids in all_modified_input_ids])
for modified_input_ids in all_modified_input_ids:
  padding = max_length - modified_input_ids.shape[1]
  padded_tensor = torch.cat([torch.full((1, padding), 128263, dtype = torch.int64), modified_input_ids], dim = 1)
  attention_mask = torch.cat([torch.zeros((1, padding), dtype = torch.int64), torch.ones((1, modified_input_ids.shape[1]), dtype = torch.int64)], dim = 1)
  all_padded_tensors.append(padded_tensor)
  all_attention_masks.append(attention_mask)

all_padded_tensors = torch.cat(all_padded_tensors, dim = 0)
all_attention_masks = torch.cat(all_attention_masks, dim = 0)

input_ids = all_padded_tensors.to("cuda")
attention_mask = all_attention_masks.to("cuda")
generated_ids = model.generate(
      input_ids = input_ids,
      attention_mask = attention_mask,
      max_new_tokens = 1200,
      do_sample = True,
      temperature = 0.6,
      top_p = 0.95,
      repetition_penalty = 1.1,
      num_return_sequences = 1,
      eos_token_id = 128258,
     use_cache = True
  )
token_to_find = 128257   # Start of speech
token_to_remove = 128258 # End of speech

# Crop per-row (not per-batch): when a reference-audio prefix is used, its own
# Start-of-speech token would otherwise throw off a single batch-wide crop index.
code_lists = []
for row in generated_ids:
    hits = (row == token_to_find).nonzero(as_tuple = True)[0]
    row = row[hits[-1].item() + 1:] if len(hits) > 0 else row
    stop = (row == token_to_remove).nonzero(as_tuple = True)[0]
    if len(stop) > 0:
        row = row[: stop[0].item()]
    row = row[row >= 128266] # drop any stray special/pad tokens
    row_length = row.size(0)
    new_length = (row_length // 7) * 7
    trimmed_row = row[:new_length]
    trimmed_row = [t.item() - 128266 for t in trimmed_row]
    code_lists.append(trimmed_row)


def redistribute_codes(code_list):
  layer_1 = []
  layer_2 = []
  layer_3 = []
  for i in range((len(code_list)+1)//7):
    layer_1.append(code_list[7*i])
    layer_2.append(code_list[7*i+1]-4096)
    layer_3.append(code_list[7*i+2]-(2*4096))
    layer_3.append(code_list[7*i+3]-(3*4096))
    layer_2.append(code_list[7*i+4]-(4*4096))
    layer_3.append(code_list[7*i+5]-(5*4096))
    layer_3.append(code_list[7*i+6]-(6*4096))
  codes = [torch.tensor(layer_1).unsqueeze(0),
         torch.tensor(layer_2).unsqueeze(0),
         torch.tensor(layer_3).unsqueeze(0)]

  # codes = [c.to("cuda") for c in codes]
  audio_hat = snac_model.decode(codes)
  return audio_hat

my_samples = []
for code_list in code_lists:
  samples = redistribute_codes(code_list)
  my_samples.append(samples)
from IPython.display import display, Audio
if len(prompts) != len(my_samples):
  raise Exception("Number of prompts and samples do not match")
else:
  for i in range(len(my_samples)):
    print(prompts[i])
    samples = my_samples[i]
    display(Audio(samples.detach().squeeze().to("cpu").numpy(), rate = 24000))
# Clean up to save RAM
del my_samples,samples

วันนี้เป็นวันที่ดีมากมากเลย สวัสดีครับผม


Ib Os Peb


In [ ]:
#@title Did the tags survive the fine-tune? Compare against the base checkpoint

# If the fine-tuned row speaks the tag out loud (or ignores it) while the base row performs
# it, the LoRA has drifted: raise TAG_MIX_RATIO / DISTIL_PER_TAG, or lower learning_rate.
from IPython.display import display, Audio

TAG_CHECK = [
    ("ผมเหนื่อยมากเลยวันนี้ <sigh> ขอพักก่อนนะ",        None),
    ("เรื่องนี้ขำมากเลย <laugh> เล่าอีกทีได้ไหม",          None),
    ("That is hilarious. <laugh> Tell me another one.", "tara"),
]

snac_model.to("cuda")
for text, base_voice in TAG_CHECK:
    print(text, "| tags:", scan_tags(text)[0])
    runs = [("fine-tuned", None, False)]
    if base_voice:
        runs.append((f"base checkpoint ({base_voice})", base_voice, True))
    for label, voice, use_base in runs:
        codes, finished = orpheus_generate_codes([text], voice = voice, use_base = use_base)[0]
        if not codes_are_valid(codes):
            print(f"  {label}: output was not decodable, sample again")
            continue
        print(f"  {label} - {(len(codes) // 7) / SNAC_FRAME_RATE:.1f}s"
              + ("" if finished else " (hit max_new_tokens)"))
        display(Audio(codes_to_audio(codes), rate = 24000))


ผมเหนื่อยมากเลยวันนี้ <sigh> ขอพักก่อนนะ | tags: ['<sigh>']
  fine-tuned - 4.1s


เรื่องนี้ขำมากเลย <laugh> เล่าอีกทีได้ไหม | tags: ['<laugh>']
  fine-tuned - 4.1s


That is hilarious. <laugh> Tell me another one. | tags: ['<laugh>']
  fine-tuned - 3.3s


  base checkpoint (tara) - 5.5s


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("orpheus_lora")  # Local saving
tokenizer.save_pretrained("orpheus_lora")
# model.push_to_hub("your_name/orpheus_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/orpheus_lora", token = "YOUR_HF_TOKEN") # Online saving

('orpheus_lora/tokenizer_config.json',
 'orpheus_lora/special_tokens_map.json',
 'orpheus_lora/chat_template.jinja',
 'orpheus_lora/tokenizer.json')

In [ ]:
# Save trained model output
# !cp -r /content/orpheus_output /content/drive/MyDrive/


### Saving to float16

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
model.save_pretrained_merged("orpheus_model_16bit", tokenizer, save_method = "merged_16bit")
# model.save_pretrained_merged("/content/drive/MyDrive/orpheus_model_16bit", tokenizer, save_method = "merged_16bit")

# Merge to 4bit
if False: model.save_pretrained_merged("orpheus_model_4bit", tokenizer, save_method = "merged_4bit")


Detected local model directory: /content/hmongTTS1
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:03<00:03,  3.53s/it]

Copied model-00002-of-00002.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:19<00:00,  9.93s/it]


Copied model-00001-of-00002.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:37<00:00, 18.80s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/hmongTTS4`


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>